# Stemming and Lemmatization

#### 1. Given the list of pluralized words below, define your own simple word stemmer function or class,  limited to only simple rules and regex. No libraries! It should strip basic endings.

In [159]:
plurals = [
    "flies",
    "denied",
    "itemization",
    "sensational",
    "reference",
    "colonizer",
]

def stem(word):
    # Define basic rules for stripping endings
    rules = {
        'es': '',
        's': '',
        'ed': '',
        'izer': '',
        'ization': '',
        'al': '',
        'ence': '',
        'ing': '',
        'ly': '',
        'ful': '',
    }

    for ending, replacement in rules.items():
        if word.endswith(ending):
            return word[:-len(ending)]
    
    return word

# Stem each word in the list
for plural in plurals:
    print(plural + " -> " + stem(plural))

flies -> fli
denied -> deni
itemization -> item
sensational -> sensation
reference -> refer
colonizer -> colon


#### 2. After your initial implementation, run it on the following words:

In [160]:
new_words = [
    "friendly",
    "puzzling",
    "helpful",
]

# Stem each word in the list
for words in new_words:
    print(words + " -> " + stem(words))

friendly -> friend
puzzling -> puzzl
helpful -> help


#### 3. Realizing that fixing future words manually can be problematic, use a desired NLTK stemmer and run it on all the words:

In [161]:
import nltk

all_words = plurals + new_words

# Initialize the SnowballStemmer
stemmer = nltk.SnowballStemmer('english')

# Stem all words in the `all_words` list
stemmed_words = [stemmer.stem(word) for word in all_words]

print(stemmed_words)

['fli', 'deni', 'item', 'sensat', 'refer', 'colon', 'friend', 'puzzl', 'help']


#### 4. There are likely a few words in the outputs above that would cause issues in real-world applications. Pick some examples, and show how they are solved with a lemmatizer. Use either spaCy or nltk.

Your answer here! Code below.

In [162]:
import spacy

# Load the English language model in spaCy
nlp = spacy.load("en_core_web_sm")

# Examples of words that might cause issues with stemming
problematic_words = ["flies", "denied", "puzzling", "sensational"]

# Lemmatize the problematic words using spaCy
lemmatized_words = [word.lemma_ for word in nlp(' '.join(problematic_words))]

# Print the lemmatized words
print(lemmatized_words)

['fly', 'deny', 'puzzle', 'sensational']


# Stemming/Lemmatization - Practical Example
Using the news corpus (subset/category of the Brown corpus), perform common text normalization techniques such as stopword filtering and stemming/lemmatization. Compare the top 10 most common **words** before and after these normalization techniques.

In [163]:
import nltk
from nltk.corpus import brown
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from collections import Counter
import re

# T top 10 most common words before applying text normalization techniques

nltk.download('brown')  # ensure we have the data

news = brown.words(categories='news')

# create a counter object for the frequency distribution of words
word_counter = Counter(word for word in news if word.isalnum())

# print the top 10 most common words
print("Top 10 most common words in the news corpus:")
for word, freq in word_counter.most_common(10):
    print(word, freq)

[nltk_data] Downloading package brown to /home/jens/nltk_data...
[nltk_data]   Package brown is already up-to-date!


Top 10 most common words in the news corpus:
the 5580
of 2849
and 2146
to 2116
a 1993
in 1893
for 943
The 806
that 802
is 732


In [164]:
# top 10 most common words after applying text normalization techniques

nltk.download('stopwords')

# create a set of stopwords and a lemmatizer object
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# apply text normalization techniques to the news corpus
normalized_news = [lemmatizer.lemmatize(word.lower()) for word in news if word.lower() not in stop_words and word.isalpha()]

# create a counter object for the frequency distribution of normalized words
normalized_counter = nltk.Counter(normalized_news)

# print the top 10 most common normalized words
print("\nTop 10 most common normalized words in the news corpus:")
for word, freq in normalized_counter.most_common(10):
    print(word, freq)

[nltk_data] Downloading package stopwords to /home/jens/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



Top 10 most common normalized words in the news corpus:
said 406
would 246
year 244
new 241
one 221
state 213
last 177
two 174
first 158
president 143


# TF-IDF
TF-IDF (term frequency-inverse document frequency) is a way to measure the importance of a word in a document.

$$
\text{tf-idf}(t, d, D) = \text{tf}(t, d) \times \text{idf}(t, D)
$$

Where:
- $t$ is the term (word)
- $d$ is the document
- $D$ is the corpus



#### 1. Implement TF-IDF using NLTKs FreqDist (no use of e.g. scikit-learn and other high-level libraries).

In [165]:
from typing import List
from nltk.probability import FreqDist
import math


def tf(document: List[str], term: str) -> float:
    """
    Calculate the term frequency (TF) of a given term in a document.

    Args:
        document (List[str]): The document in which to calculate the term frequency.
        term (str): The term for which to calculate the term frequency.

    Returns:
        float: The term frequency of the given term in the document.
    """
    fdist = FreqDist(document)
    return fdist[term]/sum(fdist.values())


def idf(documents: List[List[str]], term: str) -> float:
    """
    Calculate the inverse document frequency (IDF) of a term in a collection of documents.

    Args:
        documents (List[List[str]]): A list of documents, where each document is represented as a list of strings.
        term (str): The term for which IDF is calculated.

    Returns:
        float: The IDF value of the term.
    """
    document_count = sum(1 for doc in documents if term in doc)
    total_documents = len(documents)
    return math.log(total_documents / document_count) + 1 if document_count != 0 else 1  # Avoid division by zero


def tf_idf(
    all_documents: List[List[str]],
    document: List[str],
    term: str,
) -> float:
    
    tf_score = tf(document, term)
    idf_score = idf(all_documents, term)
    return tf_score * idf_score

# Example usage:
corpus = [
    ["the", "quick", "brown", "fox"],
    ["jumped", "over", "the", "lazy", "dog"],
    ["the", "quick", "fox", "jumped"],
]

# List of unique terms in the corpus
unique_terms = set(term for doc in corpus for term in doc)

# Calculate TF-IDF score for each term in the corpus
for term in unique_terms:
    tfidf_scores = [tf_idf(corpus, doc, term) for doc in corpus]
    print(f"TF-IDF scores for term '{term}': {tfidf_scores}")

TF-IDF scores for term 'jumped': [0.0, 0.2810930216216329, 0.3513662770270411]
TF-IDF scores for term 'over': [0.0, 0.41972245773362205, 0.0]
TF-IDF scores for term 'dog': [0.0, 0.41972245773362205, 0.0]
TF-IDF scores for term 'fox': [0.3513662770270411, 0.0, 0.3513662770270411]
TF-IDF scores for term 'lazy': [0.0, 0.41972245773362205, 0.0]
TF-IDF scores for term 'the': [0.25, 0.2, 0.25]
TF-IDF scores for term 'quick': [0.3513662770270411, 0.0, 0.3513662770270411]
TF-IDF scores for term 'brown': [0.5246530721670275, 0.0, 0.0]


#### 2. With your TF-IDF function in place, calculate the TF-IDF for the following words in the first document of the news articles found in the Brown corpus: 

- *the*
- *nevertheless*
- *highway*
- *election*

Perform any preprocessing steps you deem necessary. Comment on your findings.

In [173]:
from nltk.corpus import brown
import string

fileids = brown.fileids(categories='news')
first_doc = list(brown.words(fileids[0]))
all_docs = [list(brown.words(fileid)) for fileid in fileids]

# Preprocessing function: remove lowercase, punctuation and empty strings
def preprocess(text):
    text = [word.lower() for word in text]
    text = [''.join(char for char in word if char not in string.punctuation) for word in text]
    text = [word for word in text if word]
    return text

# Words to calculate TF-IDF for
words_to_calculate = ['the', 'nevertheless', 'highway', 'election']

# Calculate TF-IDF scores for each word
for term in words_to_calculate:
    preprocessed_doc = preprocess(first_doc)
    tfidf_score = tf_idf(all_docs, preprocessed_doc, term)
    print(f"TF-IDF score for term '{term}': {tfidf_score}")

TF-IDF score for term 'the': 0.07796780684104627
TF-IDF score for term 'nevertheless': 0.0017092028535203073
TF-IDF score for term 'highway': 0.01297738501848645
TF-IDF score for term 'election': 0.017476088316367714


#### 3. While TF-IDF is primarily used for information retrieval and text mining, reflect on how TF-IDF could be used in a language modeling context.

Utilizing TF-IDF to assess the importance of words within a document or corpus could  enhance interpretability, particularly by prioritizing relevant terms for tasks like text summarization and keyword extraction, effectively highlighting the distinct nature of the document.

#### 4. You were previously introduced to word representations. TF-IDF can be considered one. What are some differences between the TF-IDF output and one that is computed once from a vocabulary (e.g. one-hot encoding)?

TF-IDF considers both term frequency and corpus-wide importance, whereas one-hot encoding represents the presence or absence of terms using binary vectors, which often results in high dimensionality and sparsity issues due to the lack of frequency-based weighting.

# TF-IDF - Practical Example
You will again be looking at specific words for a document, but this time weighted by their TF-IDF scores. Ideally, the scoring should be able to retrieve representative words for this document in context of its document collection or category.

You will do the following:
- Select a category from the Reuters (news) corpus
- Perform preprocessing
- Calculate TF-IDF scores
- Find the top 5 words for *each document* in a subset of documents in your collection (e.g. 5, 10, ... documents total)
- Inspect whether these words make sense for a given document, and comment on your findings.

In [174]:
import nltk; nltk.download("reuters")
from nltk.corpus import reuters

# Download the Reuters corpus
nltk.download("reuters")
nltk.download('stopwords')

# Select a category from the Reuters corpus
category = "tea"

# Create a set of stopwords and a lemmatizer object
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Get fileids for documents in the selected category
fileids = reuters.fileids(category)

# Calculate TF-IDF scores for each document in a subset
subset_size = 5
subset_fileids = fileids[:subset_size]

def extract_ngrams(tokens, n=2):
    ngrams = []
    for i in range(len(tokens) - n + 1):
        ngrams.append(tuple(tokens[i:i+n]))
    return ngrams

# Function to preprocess documents
def custom_preprocessing(document):
    tokens = [lemmatizer.lemmatize(word.lower()) for word in document if word.isalnum() and word.lower() not in stop_words]
    ngrams = extract_ngrams(tokens)
    return ngrams

for fileid in subset_fileids:
    documents = reuters.words(fileid)
    preprocessed_doc = custom_preprocessing(documents)

    corpus = [custom_preprocessing(reuters.words(fid)) for fid in fileids]

    tfidf_scores = []
    for term in set(preprocessed_doc):
        score = tf_idf(corpus, preprocessed_doc, term)
        tfidf_scores.append((term, score))
    
    tfidf_scores.sort(key=lambda x: x[1], reverse=True)

    top_words = [word for word, score in tfidf_scores[:5]]
    print(f"Top 5 words for document '{fileid}': {top_words}")

[nltk_data] Downloading package reuters to /home/jens/nltk_data...
[nltk_data]   Package reuters is already up-to-date!
[nltk_data] Downloading package reuters to /home/jens/nltk_data...
[nltk_data]   Package reuters is already up-to-date!
[nltk_data] Downloading package stopwords to /home/jens/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Top 5 words for document 'test/16225': [('tea', 'import'), ('told', 'reuters'), ('corporate', 'law'), ('cla', 'would'), ('import', 'investigation')]
Top 5 words for document 'test/17494': [('tea', 'worker'), ('violent', 'protest'), ('procedure', 'cwc'), ('launch', 'one'), ('worker', 'indian')]
Top 5 words for document 'test/19672': [('export', 'earnings'), ('annual', 'export'), ('foreign', 'exchange'), ('rbi', 'said'), ('export', 'promotion')]
Top 5 words for document 'test/19982': [('pct', 'pakistan'), ('tea', 'import'), ('trader', 'said'), ('70', 'pct'), ('suspended', 'import')]
Top 5 words for document 'training/10268': [('source', 'said'), ('trade', 'source'), ('countertrade', 'deal'), ('official', 'said'), ('billion', 'dlrs')]


* Inspect whether these words make sense for a given document, and comment on your findings.

In the context of the category "tea", most of the 25 terms make sense as they seem directly relevant to trade, production and business relations. It was necessary to filter the words for better contextual relevance, the resulting tokens seem plausible. (before extracting the digrams only the latter half of the subset of documents made sense given the cetegory context.)

# Part-of-speech tagging

#### 1. Briefly describe your understanding of POS tagging and its possible use-cases in context of text generation applications/language modeling.

POS tagging involves labeling words in a sentence with their respective parts of speech, aiding language models in generating grammatically correct and contextually relevant text

#### 2. Train a UnigramTagger (NLTK) using the Brown corpus. 
Hint: the taggers in nltk require a list of sentences containing tagged words.

In [168]:
import nltk
from nltk.corpus import brown

# Download the Brown corpus if not already downloaded
nltk.download('brown')

# Get tagged sentences from the Brown corpus
tagged_sentences = brown.tagged_sents()

# Train a UnigramTagger on the tagged sentences
unigram_tagger = nltk.UnigramTagger(tagged_sentences)

def tagger(text):
    return unigram_tagger.tag(nltk.word_tokenize(text))

# Example usage: tagging a sentence
sentence = "The quick brown fox jumps over the lazy dog"
tagged_sentence = tagger(sentence)
print(tagged_sentence)


[nltk_data] Downloading package brown to /home/jens/nltk_data...
[nltk_data]   Package brown is already up-to-date!


[('The', 'AT'), ('quick', 'JJ'), ('brown', 'JJ'), ('fox', 'NN'), ('jumps', 'NNS'), ('over', 'IN'), ('the', 'AT'), ('lazy', 'JJ'), ('dog', 'NN')]


#### 3. Use this tagger to tag the text given below. Print out the POS tags for all variants of "justify"

In [169]:
### TODO make a function as tagger?
import nltk

text = """
Imagine a situation where you have to explain why you did something – that's when you justify your actions. So, let's say you made a decision; you, as the justifier, need to give good reasons (justifications) for your choice. You might use justifying words to make your point clear and reasonable. Justifying can be a bit like saying, "Here's why I did what I did." When you justify things, you're basically providing the why behind your actions. So, being a good justifier involves carefully explaining, giving reasons, and making sure others understand your choices
"""

# Tokenize the text into sentences
sentences = nltk.sent_tokenize(text)

# Tokenize each sentence into words and tag with the UnigramTagger
tagged_text = []
for sentence in sentences:
    tagged_words = tagger(sentence)
    tagged_text.extend(tagged_words)

# Find and print POS tags for all variants of "justify"
variants = ["justify", "justifier", "justifying", "justifications"]
for variant in variants:
    pos_tags = [tag for word, tag in tagged_text if word.lower() == variant]
    print(f"POS tags for '{variant}': {pos_tags}")

POS tags for 'justify': ['VB', 'VB']
POS tags for 'justifier': [None, None]
POS tags for 'justifying': ['VBG', None]
POS tags for 'justifications': ['NNS']


#### 4. Your results may be disappointing. Repeat the same task as above using both the default NLTK pos-tagger and with spaCy. Compare the results

In [170]:
import nltk
nltk.download('averaged_perceptron_tagger')

# Download NLTK resources if not already downloaded
nltk.download('punkt')

# Sample text
text = """
Imagine a situation where you have to explain why you did something – that's when you justify your actions. So, let's say you made a decision; you, as the justifier, need to give good reasons (justifications) for your choice. You might use justifying words to make your point clear and reasonable. Justifying can be a bit like saying, "Here's why I did what I did." When you justify things, you're basically providing the why behind your actions. So, being a good justifier involves carefully explaining, giving reasons, and making sure others understand your choices
"""

# default NLTK POS tagging
tokens = nltk.word_tokenize(text)
nltk_tagged = nltk.pos_tag(tokens)

print("NLTK POS tagging:")
print(nltk_tagged)

NLTK POS tagging:
[('Imagine', 'VB'), ('a', 'DT'), ('situation', 'NN'), ('where', 'WRB'), ('you', 'PRP'), ('have', 'VBP'), ('to', 'TO'), ('explain', 'VB'), ('why', 'WRB'), ('you', 'PRP'), ('did', 'VBD'), ('something', 'NN'), ('–', 'NN'), ('that', 'WDT'), ("'s", 'VBZ'), ('when', 'WRB'), ('you', 'PRP'), ('justify', 'VBP'), ('your', 'PRP$'), ('actions', 'NNS'), ('.', '.'), ('So', 'RB'), (',', ','), ('let', 'VB'), ("'s", 'POS'), ('say', 'VB'), ('you', 'PRP'), ('made', 'VBD'), ('a', 'DT'), ('decision', 'NN'), (';', ':'), ('you', 'PRP'), (',', ','), ('as', 'IN'), ('the', 'DT'), ('justifier', 'NN'), (',', ','), ('need', 'VBP'), ('to', 'TO'), ('give', 'VB'), ('good', 'JJ'), ('reasons', 'NNS'), ('(', '('), ('justifications', 'NNS'), (')', ')'), ('for', 'IN'), ('your', 'PRP$'), ('choice', 'NN'), ('.', '.'), ('You', 'PRP'), ('might', 'MD'), ('use', 'VB'), ('justifying', 'VBG'), ('words', 'NNS'), ('to', 'TO'), ('make', 'VB'), ('your', 'PRP$'), ('point', 'NN'), ('clear', 'JJ'), ('and', 'CC'), ('rea

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/jens/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt to /home/jens/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [171]:
import spacy

# spaCy POS tagging
nlp = spacy.load("en_core_web_sm")
spacy_doc = nlp(text)

spacy_tagged = [(token.text, token.pos_) for token in spacy_doc]

print("\nspaCy POS tagging:")
print(spacy_tagged)


spaCy POS tagging:
[('\n', 'SPACE'), ('Imagine', 'VERB'), ('a', 'DET'), ('situation', 'NOUN'), ('where', 'SCONJ'), ('you', 'PRON'), ('have', 'VERB'), ('to', 'PART'), ('explain', 'VERB'), ('why', 'SCONJ'), ('you', 'PRON'), ('did', 'VERB'), ('something', 'PRON'), ('–', 'PUNCT'), ('that', 'PRON'), ("'s", 'AUX'), ('when', 'SCONJ'), ('you', 'PRON'), ('justify', 'VERB'), ('your', 'PRON'), ('actions', 'NOUN'), ('.', 'PUNCT'), ('So', 'ADV'), (',', 'PUNCT'), ('let', 'VERB'), ("'s", 'PRON'), ('say', 'VERB'), ('you', 'PRON'), ('made', 'VERB'), ('a', 'DET'), ('decision', 'NOUN'), (';', 'PUNCT'), ('you', 'PRON'), (',', 'PUNCT'), ('as', 'ADP'), ('the', 'DET'), ('justifier', 'NOUN'), (',', 'PUNCT'), ('need', 'VERB'), ('to', 'PART'), ('give', 'VERB'), ('good', 'ADJ'), ('reasons', 'NOUN'), ('(', 'PUNCT'), ('justifications', 'NOUN'), (')', 'PUNCT'), ('for', 'ADP'), ('your', 'PRON'), ('choice', 'NOUN'), ('.', 'PUNCT'), ('You', 'PRON'), ('might', 'AUX'), ('use', 'VERB'), ('justifying', 'VERB'), ('words',

Comparing the results Its evident that:

#### NTLK:
* NTLK treats contractions like single tokens: "You're" and "Here's". 
* NTLK doesn't handle punctuation more than as a tag of itself except for rare circumstances: "." to ".", however ";" to ":" and "-" as a part of the previous token ("something").
* NTLK maintains a higher level of accuracy as it uses more tags to differentiate different states of the tokens: personal pronoun, plural nouns.

#### Spacy:
* Spacy manages to extract the contractions into its two base words: "you are" and "here is".
* Spacy tokenizes with a tag "punct" for all punctuation tokens.
* Spacy uses more direct tagging where the form is the same for all token states.

The differences comes down to tokenization, tagging accuracy, and handling of special characters between NLTK and spaCy taggers.

#### 5. Finally, explore more features of the what the spaCy *document* includes related to topics covered in this lab.

In [172]:
import spacy

# spaCy POS tagging
nlp = spacy.load("en_core_web_sm")
spacy_doc = nlp(text)

spacy_tagged = [(token.text, token.lemma_, token.is_alpha, token.is_stop) for token in spacy_doc]

print("\nspaCy features")
for item in spacy_tagged:
    print(item)  # Print the first element of each tuple


spaCy features
('\n', '\n', False, False)
('Imagine', 'imagine', True, False)
('a', 'a', True, True)
('situation', 'situation', True, False)
('where', 'where', True, True)
('you', 'you', True, True)
('have', 'have', True, True)
('to', 'to', True, True)
('explain', 'explain', True, False)
('why', 'why', True, True)
('you', 'you', True, True)
('did', 'do', True, True)
('something', 'something', True, True)
('–', '–', False, False)
('that', 'that', True, True)
("'s", 'be', False, True)
('when', 'when', True, True)
('you', 'you', True, True)
('justify', 'justify', True, False)
('your', 'your', True, True)
('actions', 'action', True, False)
('.', '.', False, False)
('So', 'so', True, True)
(',', ',', False, False)
('let', 'let', True, False)
("'s", 'us', False, True)
('say', 'say', True, True)
('you', 'you', True, True)
('made', 'make', True, True)
('a', 'a', True, True)
('decision', 'decision', True, False)
(';', ';', False, False)
('you', 'you', True, True)
(',', ',', False, False)
('as'